Creating a pipeline

In [31]:
import os
import pandas as pd
import numpy as np
import nltk
import spacy
from scipy.stats import ttest_ind

from textblob import TextBlob
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from nltk.tokenize import word_tokenize
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression


Module 1. Content Performance Tracker (Data Extraction Engine)


In [23]:

df = pd.read_csv("Cleaned_Viral_Social_Media_Trends.csv")

print("Shape:", df.shape)
print(df.head())
print("\nMissing values:\n", df.isnull().sum())
print("\nDuplicates:", df.duplicated().sum())

df["Post_Date"] = pd.to_datetime(df["Post_Date"], errors="coerce")

num_cols = ["Views", "Likes", "Shares", "Comments"]

for col in num_cols:
    df[col] = pd.to_numeric(df[col], errors="coerce")

text_cols = ["Platform", "Hashtag", "Content_Type", "Region", "Engagement_Level"]

for col in text_cols:
    df[col] = df[col].astype(str).str.strip()

df["Engagement_Rate"] = np.where(
    df["Views"] > 0,
    (df["Likes"] + df["Shares"] + df["Comments"]) / df["Views"] * 100,
    0
).round(2)

df["Post_Month"] = df["Post_Date"].dt.to_period("M").astype(str)
df["Post_Year"] = df["Post_Date"].dt.year
df["Day_of_Week"] = df["Post_Date"].dt.day_name()

df = df.drop_duplicates()

print("\nFinal Shape:", df.shape)
print("\nPerformance Summary:")
print(df[num_cols + ["Engagement_Rate"]].describe())

os.makedirs("../data/processed", exist_ok=True)

df.to_csv(
    "../data/processed/Cleaned_Viral_Social_Media_Trends_processed.csv",
    index=False
)

print("\nProcessed file saved successfully!")

Shape: (5000, 11)
  Post_ID   Post_Date   Platform     Hashtag Content_Type     Region    Views  \
0  Post_1  2022-01-13     TikTok  #Challenge        Video         UK  4163464   
1  Post_2  2022-05-13  Instagram  #Education       Shorts      India  4155940   
2  Post_3  2022-01-07    Twitter  #Challenge        Video     Brazil  3666211   
3  Post_4  2022-12-05    YouTube  #Education       Shorts  Australia   917951   
4  Post_5  2023-03-23     TikTok      #Dance         Post     Brazil    64866   

    Likes  Shares  Comments Engagement_Level  
0  339431   53135     19346             High  
1  215240   65860     27239           Medium  
2  327143   39423     36223           Medium  
3  127125   11687     36806              Low  
4  171361   69581      6376           Medium  

Missing values:
 Post_ID             0
Post_Date           0
Platform            0
Hashtag             0
Content_Type        0
Region              0
Views               0
Likes               0
Shares             

Module 2: Virality Prediction Engine.

In [24]:
df = pd.read_csv(
    "../data/processed/Cleaned_Viral_Social_Media_Trends_processed.csv"
)

df = df.dropna(
    subset=[
        "Views",
        "Likes",
        "Shares",
        "Comments",
        "Hashtag"
    ]
)

df = df[df["Views"] > 0]

df["Share_Rate"] = df["Shares"] / df["Views"]

df["Like_Rate"] = df["Likes"] / df["Views"]

df["Comment_Rate"] = df["Comments"] / df["Views"]

df["Viral_Coefficient"] = (
    df["Share_Rate"] * 0.6
    + df["Comment_Rate"] * 0.2
    + df["Like_Rate"] * 0.2
)

features = [
    "Views",
    "Likes",
    "Shares",
    "Comments",
    "Share_Rate",
    "Like_Rate",
    "Comment_Rate"
]

scaler = StandardScaler()

X = scaler.fit_transform(
    df[features]
)

y = df["Viral_Coefficient"]

model = RandomForestRegressor(
    n_estimators=100,
    random_state=42
)

model.fit(X, y)

df["Predicted_Virality"] = model.predict(X)

df["Virality_Score"] = (
    df["Predicted_Virality"] * 100
).round(2)

df["Virality_Level"] = pd.qcut(
    df["Virality_Score"],
    q=3,
    labels=["Low", "Medium", "High"],
    duplicates="drop"
)

print("VIRALITY PREDICTION RESULTS")

print("\nTop 10 Viral Posts:")

print(
    df[
        [
            "Post_ID",
            "Platform",
            "Hashtag",
            "Content_Type",
            "Views",
            "Likes",
            "Shares",
            "Comments",
            "Viral_Coefficient",
            "Virality_Score",
            "Virality_Level"
        ]
    ]
    .sort_values(
        "Virality_Score",
        ascending=False
    )
    .head(10)
)

topic_results = (
    df.groupby("Hashtag")
    .agg(
        Average_Viral_Coefficient=(
            "Viral_Coefficient",
            "mean"
        ),
        Average_Virality_Score=(
            "Virality_Score",
            "mean"
        ),
        Average_Shares=(
            "Shares",
            "mean"
        ),
        Average_Views=(
            "Views",
            "mean"
        ),
        Total_Shares=(
            "Shares",
            "sum"
        ),
        Number_of_Posts=(
            "Post_ID",
            "count"
        )
    )
    .sort_values(
        "Average_Virality_Score",
        ascending=False
    )
)

print("\nTOP VIRAL TOPICS")

print(
    topic_results.head(10)
)

best_topic = topic_results.index[0]

print("\nMOST VIRAL TOPIC:")
print(best_topic)

df.to_csv(
    "../data/processed/virality_prediction_results.csv",
    index=False
)

topic_results.to_csv(
    "../data/processed/viral_topic_analysis.csv"
)

print("\nModule 2 completed successfully!")

VIRALITY PREDICTION RESULTS

Top 10 Viral Posts:
        Post_ID   Platform     Hashtag Content_Type  Views   Likes  Shares  \
4826  Post_4827     TikTok      #Viral       Shorts   1266  318849    1715   
1540  Post_1541  Instagram     #Comedy  Live Stream   4323  301571   81786   
3686  Post_3687    YouTube       #Tech  Live Stream   5467  337597   79716   
2647  Post_2648    YouTube  #Challenge         Post   8982  353646   77267   
4571  Post_4572     TikTok     #Gaming        Video   5679   95171   63079   
4691  Post_4692    Twitter  #Education       Shorts   8162  285470   60984   
1033  Post_1034  Instagram  #Education        Tweet  11338  379782   86100   
4991  Post_4992  Instagram    #Fashion       Shorts  10157  322897   93292   
4137  Post_4138  Instagram     #Comedy        Tweet  13578  485916   39122   
4284  Post_4285    YouTube     #Gaming  Live Stream   7810  277197   12486   

      Comments  Viral_Coefficient  Virality_Score Virality_Level  
4826     36121          5

Module 3: Audience Sentiment Analyzer

In [25]:
df2 = pd.read_csv("Social media_Data.csv")

print(df2.head())
print("\nShape:", df2.shape)
print("\nColumns:", df2.columns.tolist())
print("\nMissing values:")
print(df2.isnull().sum())
print("\nCategory distribution:")
print(df2["category"].value_counts())

                                       clean_comment  category
0   family mormon have never tried explain them t...         1
1  buddhism has very much lot compatible with chr...         1
2  seriously don say thing first all they won get...        -1
3  what you have learned yours and only yours wha...         0
4  for your own benefit you may want read living ...         1

Shape: (37249, 2)

Columns: ['clean_comment', 'category']

Missing values:
clean_comment    100
category           0
dtype: int64

Category distribution:
category
 1    15830
 0    13142
-1     8277
Name: count, dtype: int64


In [26]:
df2 = df2.dropna(subset=["clean_comment"])

In [27]:
nltk.download("punkt")
nltk.download("punkt_tab")
nltk.download("stopwords")
nltk.download("wordnet")

df = pd.read_csv("Social media_Data.csv")

print("Shape:", df.shape)
print(df.head())

df = df.dropna(subset=["clean_comment", "category"])

df["clean_comment"] = df["clean_comment"].astype(str).str.strip()

df = df[df["clean_comment"] != ""]

df = df.drop_duplicates(subset=["clean_comment"])

stop_words = set(stopwords.words("english"))
lemmatizer = WordNetLemmatizer()

def preprocess_text(text):
    tokens = word_tokenize(text.lower())
    tokens = [
        word for word in tokens
        if word.isalpha() and word not in stop_words
    ]
    tokens = [
        lemmatizer.lemmatize(word)
        for word in tokens
    ]
    return " ".join(tokens)

df["processed_comment"] = df["clean_comment"].apply(
    preprocess_text
)

df["Polarity"] = df["clean_comment"].apply(
    lambda x: TextBlob(x).sentiment.polarity
)

df["Subjectivity"] = df["clean_comment"].apply(
    lambda x: TextBlob(x).sentiment.subjectivity
)

def sentiment_label(score):
    if score > 0:
        return "Positive"
    elif score < 0:
        return "Negative"
    else:
        return "Neutral"

df["Sentiment"] = df["Polarity"].apply(
    sentiment_label
)

trigger_words = [
    "problem", "struggle", "difficult", "difficulty",
    "stress", "worried", "worry", "confused",
    "frustrated", "sad", "fear", "afraid",
    "lonely", "pain", "hate", "hard",
    "issue", "help", "anxious", "depressed"
]

def find_triggers(text):
    words = text.lower().split()
    found = [
        word for word in trigger_words
        if word in words
    ]
    return ", ".join(found)

df["Linguistic_Triggers"] = df["clean_comment"].apply(
    find_triggers
)

df["Problem_Awareness"] = np.where(
    df["Linguistic_Triggers"] != "",
    "Yes",
    "No"
)

df["Trigger_Count"] = df["Linguistic_Triggers"].apply(
    lambda x: len(x.split(", ")) if x else 0
)

df["Relatability"] = np.where(
    (df["Problem_Awareness"] == "Yes") |
    (df["Sentiment"] != "Neutral"),
    "Relatable",
    "Neutral"
)

print("\nSentiment Distribution:")
print(df["Sentiment"].value_counts())

print("\nProblem Awareness:")
print(df["Problem_Awareness"].value_counts())

print("\nTop Linguistic Triggers:")
print(
    df["Linguistic_Triggers"]
    .str.split(", ")
    .explode()
    .value_counts()
    .head(15)
)

print("\nFinal Results:")
print(
    df[
        [
            "clean_comment",
            "category",
            "Sentiment",
            "Polarity",
            "Subjectivity",
            "Linguistic_Triggers",
            "Trigger_Count",
            "Problem_Awareness",
            "Relatability"
        ]
    ].head(10)
)

df.to_csv(
    "audience_sentiment_results.csv",
    index=False
)

print("\nModule 3 completed successfully!")
print("Saved: audience_sentiment_results.csv")

[nltk_data] Downloading package punkt to C:\Users\Shruti
[nltk_data]     Gaikwad\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to C:\Users\Shruti
[nltk_data]     Gaikwad\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to C:\Users\Shruti
[nltk_data]     Gaikwad\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to C:\Users\Shruti
[nltk_data]     Gaikwad\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


Shape: (37249, 2)
                                       clean_comment  category
0   family mormon have never tried explain them t...         1
1  buddhism has very much lot compatible with chr...         1
2  seriously don say thing first all they won get...        -1
3  what you have learned yours and only yours wha...         0
4  for your own benefit you may want read living ...         1

Sentiment Distribution:
Sentiment
Positive    15721
Neutral     12650
Negative     8236
Name: count, dtype: int64

Problem Awareness:
Problem_Awareness
No     33608
Yes     2999
Name: count, dtype: int64

Top Linguistic Triggers:
Linguistic_Triggers
             33608
help           531
hate           527
problem        475
issue          463
hard           428
sad            259
fear           161
difficult      141
worry           89
afraid          82
worried         75
pain            50
confused        49
struggle        45
Name: count, dtype: int64

Final Results:
                          

Module 4 – A/B Testing Framework

In [28]:
df = pd.read_csv(
    "../data/processed/Cleaned_Viral_Social_Media_Trends_processed.csv"
)

df["Engagement_Rate"] = pd.to_numeric(
    df["Engagement_Rate"],
    errors="coerce"
)

df = df.dropna(
    subset=["Content_Type", "Engagement_Rate"]
)

print("Content Types:")
print(df["Content_Type"].value_counts())

results = []

content_types = df["Content_Type"].dropna().unique()

for i in range(len(content_types)):
    for j in range(i + 1, len(content_types)):

        group_a = df[
            df["Content_Type"] == content_types[i]
        ]["Engagement_Rate"]

        group_b = df[
            df["Content_Type"] == content_types[j]
        ]["Engagement_Rate"]

        t_stat, p_value = ttest_ind(
            group_a,
            group_b,
            equal_var=False
        )

        results.append({
            "Group_A": content_types[i],
            "Group_B": content_types[j],
            "Mean_A": group_a.mean(),
            "Mean_B": group_b.mean(),
            "Difference": group_a.mean() - group_b.mean(),
            "T_Statistic": t_stat,
            "P_Value": p_value,
            "Significant": "Yes" if p_value < 0.05 else "No"
        })

ab_results = pd.DataFrame(results)

print("\nA/B Testing Results:")
print(ab_results)

print("\nAverage Engagement by Content Type:")
print(
    df.groupby("Content_Type")["Engagement_Rate"]
    .mean()
    .sort_values(ascending=False)
)

ab_results.to_csv(
    "../data/processed/ab_testing_results.csv",
    index=False
)

print("\nModule 4 completed successfully!")

Content Types:
Content_Type
Live Stream    855
Post           853
Reel           841
Tweet          836
Video          828
Shorts         787
Name: count, dtype: int64

A/B Testing Results:
        Group_A      Group_B     Mean_A     Mean_B  Difference  T_Statistic  \
0         Video       Shorts  48.820676  80.628958  -31.808282    -0.851926   
1         Video         Post  48.820676  49.806495   -0.985818    -0.096032   
2         Video        Tweet  48.820676  52.069139   -3.248462    -0.293733   
3         Video  Live Stream  48.820676  61.876725  -13.056049    -0.765407   
4         Video         Reel  48.820676  48.753068    0.067609     0.007092   
5        Shorts         Post  80.628958  49.806495   30.822463     0.819761   
6        Shorts        Tweet  80.628958  52.069139   28.559819     0.755077   
7        Shorts  Live Stream  80.628958  61.876725   18.752233     0.468910   
8        Shorts         Reel  80.628958  48.753068   31.875890     0.852161   
9          Post     

Module 5 – Engagement Optimization Recommender

In [29]:
df = pd.read_csv(
    "../data/processed/viral_prediction_results.csv"
)

df["Engagement_Rate"] = pd.to_numeric(
    df["Engagement_Rate"],
    errors="coerce"
)

df["Virality_Score"] = pd.to_numeric(
    df["Virality_Score"],
    errors="coerce"
)

df = df.dropna(
    subset=[
        "Hashtag",
        "Content_Type",
        "Engagement_Rate",
        "Virality_Score"
    ]
)

topic_analysis = (
    df.groupby("Hashtag")
    .agg(
        Avg_Engagement=("Engagement_Rate", "mean"),
        Avg_Virality=("Virality_Score", "mean"),
        Total_Views=("Views", "sum"),
        Total_Shares=("Shares", "sum"),
        Posts=("Post_ID", "count")
    )
    .sort_values(
        ["Avg_Engagement", "Avg_Virality"],
        ascending=False
    )
)

content_analysis = (
    df.groupby("Content_Type")
    .agg(
        Avg_Engagement=("Engagement_Rate", "mean"),
        Avg_Virality=("Virality_Score", "mean"),
        Total_Views=("Views", "sum"),
        Total_Shares=("Shares", "sum"),
        Posts=("Post_ID", "count")
    )
    .sort_values(
        ["Avg_Engagement", "Avg_Virality"],
        ascending=False
    )
)

best_topic = topic_analysis.index[0]
best_content = content_analysis.index[0]

print("ENGAGEMENT OPTIMIZATION RECOMMENDER")

print("\nTop Topics:")
print(topic_analysis.head(10))

print("\nBest Content Types:")
print(content_analysis)

print("\nRECOMMENDATION FOR NEXT WEEK")

print("Recommended Topic:", best_topic)
print("Recommended Content Type:", best_content)

print("\nRecommended Strategy:")
print(
    f"Focus on {best_topic} content using the "
    f"{best_content} format because it has the highest "
    f"historical engagement and virality."
)

topic_analysis.to_csv(
    "../data/processed/topic_recommendations.csv"
)

content_analysis.to_csv(
    "../data/processed/content_type_recommendations.csv"
)

print("\nModule 5 completed successfully!")

ENGAGEMENT OPTIMIZATION RECOMMENDER

Top Topics:
            Avg_Engagement  Avg_Virality  Total_Views  Total_Shares  Posts
Hashtag                                                                   
#Viral          100.474782      0.489193   1172480925      24256296    481
#Comedy          66.956614      0.341816   1237321563      24956115    505
#Gaming          62.067516      0.317775   1197834795      24601832    479
#Fitness         54.414067      0.274440   1393273574      27434152    536
#Tech            53.609837      0.270966   1235543295      23709020    491
#Education       50.872114      0.258276   1328894615      27168070    525
#Fashion         48.844394      0.249282   1181866511      24816032    487
#Dance           46.663286      0.237872   1213891935      24580747    496
#Challenge       45.959093      0.233305   1242826928      25949491    507
#Music           39.549047      0.198501   1266398079      25126055    493

Best Content Types:
              Avg_Engagement  

Module 6 – Growth Visualization Dashboard

In [30]:
df = pd.read_csv(
    "../data/processed/virality_prediction_results.csv"
)

df["Post_Date"] = pd.to_datetime(
    df["Post_Date"],
    errors="coerce"
)

df["Engagement_Rate"] = pd.to_numeric(
    df["Engagement_Rate"],
    errors="coerce"
)

df["Share_Rate"] = np.where(
    df["Views"] > 0,
    df["Shares"] / df["Views"] * 100,
    0
)

df["Comment_Rate"] = np.where(
    df["Views"] > 0,
    df["Comments"] / df["Views"] * 100,
    0
)

df["Like_Rate"] = np.where(
    df["Views"] > 0,
    df["Likes"] / df["Views"] * 100,
    0
)

df["Month"] = df["Post_Date"].dt.to_period("M").astype(str)

df.to_csv(
    "../data/processed/dashboard_data.csv",
    index=False
)


print(df.head())

  Post_ID  Post_Date   Platform     Hashtag Content_Type     Region    Views  \
0  Post_1 2022-01-13     TikTok  #Challenge        Video         UK  4163464   
1  Post_2 2022-05-13  Instagram  #Education       Shorts      India  4155940   
2  Post_3 2022-01-07    Twitter  #Challenge        Video     Brazil  3666211   
3  Post_4 2022-12-05    YouTube  #Education       Shorts  Australia   917951   
4  Post_5 2023-03-23     TikTok      #Dance         Post     Brazil    64866   

    Likes  Shares  Comments  ... Post_Year  Day_of_Week  Share_Rate  \
0  339431   53135     19346  ...      2022     Thursday    1.276221   
1  215240   65860     27239  ...      2022       Friday    1.584720   
2  327143   39423     36223  ...      2022       Friday    1.075306   
3  127125   11687     36806  ...      2022       Monday    1.273162   
4  171361   69581      6376  ...      2023     Thursday  107.268831   

    Like_Rate Comment_Rate  Viral_Coefficient  Predicted_Virality  \
0    8.152610     0.464

Module 7 – Trend Forecasting

In [32]:
df = pd.read_csv(
    "../data/processed/Cleaned_Viral_Social_Media_Trends_processed.csv"
)

df["Post_Date"] = pd.to_datetime(
    df["Post_Date"],
    errors="coerce"
)

df = df.dropna(
    subset=["Post_Date", "Hashtag"]
)

df["Month"] = df["Post_Date"].dt.to_period("M")

monthly = (
    df.groupby(["Month", "Hashtag"])
    .size()
    .reset_index(name="Post_Count")
)

results = []

for hashtag in monthly["Hashtag"].unique():

    data = monthly[
        monthly["Hashtag"] == hashtag
    ].sort_values("Month")

    if len(data) < 3:
        continue

    data = data.copy()

    data["Time"] = np.arange(len(data))

    X = data[["Time"]]
    y = data["Post_Count"]

    model = LinearRegression()
    model.fit(X, y)

    trend = model.coef_[0]

    future_time = np.array(
        [[len(data)]]
    )

    predicted_posts = model.predict(
        future_time
    )[0]

    results.append({
        "Hashtag": hashtag,
        "Current_Posts": data["Post_Count"].iloc[-1],
        "Average_Posts": data["Post_Count"].mean(),
        "Trend_Slope": trend,
        "Predicted_Next_Month": max(
            0,
            round(predicted_posts, 2)
        )
    })

trend_results = pd.DataFrame(results)

trend_results["Trend_Percentage"] = (
    trend_results["Trend_Slope"]
    / trend_results["Average_Posts"]
    * 100
).round(2)

trend_results["Trend_Level"] = np.where(
    trend_results["Trend_Percentage"] > 10,
    "Rising",
    np.where(
        trend_results["Trend_Percentage"] < -10,
        "Declining",
        "Stable"
    )
)

trend_results = trend_results.sort_values(
    "Trend_Percentage",
    ascending=False
)

print("TREND FORECASTING RESULTS")

print("\nTop Rising Hashtags:")

print(
    trend_results[
        trend_results["Trend_Level"] == "Rising"
    ].head(10)
)

print("\nComplete Trend Analysis:")

print(
    trend_results.head(20)
)

rising_topics = trend_results[
    trend_results["Trend_Level"] == "Rising"
]

print("\nRECOMMENDED UPCOMING TOPICS:")

print(
    rising_topics[
        [
            "Hashtag",
            "Trend_Percentage",
            "Predicted_Next_Month"
        ]
    ].head(10)
)

trend_results.to_csv(
    "../data/processed/trend_forecasting_results.csv",
    index=False
)

print("\nModule 7 completed successfully!")
print("Saved: trend_forecasting_results.csv")

c:\Users\Shruti Gaikwad\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\utils\validation.py:2827: UserWarning: X does not have valid feature names, but LinearRegression was fitted with feature names
  warnings.warn(
c:\Users\Shruti Gaikwad\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\utils\validation.py:2827: UserWarning: X does not have valid feature names, but LinearRegression was fitted with feature names
  warnings.warn(
c:\Users\Shruti Gaikwad\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\utils\validation.py:2827: UserWarning: X does not have valid feature names, but LinearRegression was fitted with feature names
  warnings.warn(
c:\Users\Shruti Gaikwad\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\utils\validation.py:2827: UserWarning: X does not have valid feature names, but LinearRegression was fitted with feature names
  warnings.warn(
c:\Users\Shruti Gaikwad\AppData\Local\Programs\Python\Python312\

TREND FORECASTING RESULTS

Top Rising Hashtags:
Empty DataFrame
Columns: [Hashtag, Current_Posts, Average_Posts, Trend_Slope, Predicted_Next_Month, Trend_Percentage, Trend_Level]
Index: []

Complete Trend Analysis:
      Hashtag  Current_Posts  Average_Posts  Trend_Slope  \
9      #Viral             26      20.041667     0.227391   
1     #Comedy             24      21.041667     0.174348   
2      #Dance             24      20.666667     0.160870   
5    #Fitness             23      22.333333     0.114783   
6     #Gaming             19      19.958333    -0.000435   
0  #Challenge             22      21.125000    -0.006522   
4    #Fashion             20      20.291667    -0.036087   
7      #Music             12      20.541667    -0.094348   
3  #Education              9      21.875000    -0.196957   
8       #Tech             22      20.458333    -0.250000   

   Predicted_Next_Month  Trend_Percentage Trend_Level  
9                 22.88              1.13      Stable  
1           